In [ ]:
import numpy as np
import pandas as pd
import yaml
import re
import json

from pandarallel import pandarallel
pandarallel.initialize(progress_bar=True)

In [ ]:
df_filtered = pd.read_csv("03_WOS_unique_orgs_firms_US_filtered.csv")
df_filtered

In [ ]:
category_all = df_filtered[df_filtered.hq_match == 1][['organization', 'country']]
category_all

In [ ]:
len(category_all.organization.unique())

## Standardized firm name (because we already excluded MNCs in the US in Post5b)

In [ ]:
from unicodedata import normalize as ucnorm

# --- Config ------------------------------------------------------------------
LEGAL_SUFFIXES = {
    "inc","incorporated","corp","corporation","co","company","ltd","llc","plc",
    "lp","llp","pc","pc.","pte","ag","gmbh","nv","sa","sas","spa","bv","sarl","sro","grp", "&", "the"
}

GENERIC_TAILS = {
    "biotech","biotechnology","biotechnol","sciences","science","scientific",
    "consulting","solutions","services","service","systems","system",
    "technologies","technology","tech","laboratories","laboratory","labs","lab",
    "institute","institutes","center","centre","group","department","division","unit",
    "partners","associates","holdings","ventures","industries","industry","values"
}

US_MARKERS = {"us","u.s","u.s.","usa","u.s.a"}

# --- Helpers -----------------------------------------------------------------
def to_ascii(s: str) -> str:
    return ucnorm("NFKD", s).encode("ascii", "ignore").decode("ascii", "ignore")

def split_tokens_keep_case(s: str):
    s = s.replace("&", " & ")
    s = re.sub(r"[;/|,_\-]+", " ", s)
    s = re.sub(r"\.", " ", s)
    return [t for t in re.findall(r"[A-Za-z0-9&]+", s) if t]

def first_segment(s: str) -> str:
    return re.split(r";", s, maxsplit=1)[0]

def strip_legal_and_us(tokens_case, tokens_lc):
    kept = []
    for orig, lc in zip(tokens_case, tokens_lc):
        if lc in LEGAL_SUFFIXES or lc in US_MARKERS:
            continue
        kept.append((orig, lc))
    return kept

def strip_trailing_generic(tokens_case, tokens_lc):
    j = len(tokens_lc)
    while j > 0 and tokens_lc[j-1] in GENERIC_TAILS:
        j -= 1
    return tokens_case[:j], tokens_lc[:j]

def normalize_row_name(name: str):
    """Return normalized tokens OR fallback cleaning if tokens vanish."""
    if not isinstance(name, str) or not name.strip():
        return {"base_tokens": [], "base_tokens_orig": [], "has_caps_brand": False, "ampersand": False, "fallback": True}

    seg = first_segment(to_ascii(name))
    toks_case = split_tokens_keep_case(seg)      # preserve case
    toks_lc   = [t.lower() for t in toks_case]

    ampersand = any("&" in t for t in toks_case)

    # --- strip legal + US
    kept = strip_legal_and_us(toks_case, toks_lc)
    kept_case, kept_lc = zip(*kept) if kept else ([], [])

    # --- then strip generic tails
    stripped_case, stripped_lc = strip_trailing_generic(list(kept_case), list(kept_lc))

    if not stripped_case:  # fallback: only legal/US stripping
        base_tokens = list(kept_lc)
        base_tokens_orig = list(kept_case)
        fallback = True
    else:
        base_tokens = stripped_lc
        base_tokens_orig = stripped_case
        fallback = False

    # uppercase preference: majority rule
    caps_count = sum(1 for tok in base_tokens_orig if tok.isalpha() and len(tok) >= 2 and tok.isupper())
    has_caps_brand = caps_count > (len(base_tokens_orig) / 2)

    return {"base_tokens": base_tokens, "base_tokens_orig": base_tokens_orig,
            "has_caps_brand": has_caps_brand, "ampersand": ampersand, "fallback": fallback}

def build_canonical_from_tokens(base_tokens, prefer_caps=False, ampersand=False):
    if not base_tokens:
        return ""
    if prefer_caps:
        pretty = " ".join(base_tokens).upper()
    else:
        words = []
        for w in base_tokens:
            if w.isalpha() and len(w) <= 4 and w.upper() == w:
                words.append(w.upper())
            else:
                words.append(w.title())
        pretty = " ".join(words)
    if ampersand:
        pretty = re.sub(r"\bAnd\b", "&", pretty)
    return pretty

# --- Main --------------------------------------------------------------------
def standardize_affiliations(df: pd.DataFrame):
    work = df.copy()

    norm = work["organization"].apply(normalize_row_name)
    work["__base_tokens"] = norm.apply(lambda d: d["base_tokens"])
    work["__fallback"]    = norm.apply(lambda d: d["fallback"])
    work["__norm_key"]    = norm.apply(lambda d: "".join(d["base_tokens"]))
    work["__has_caps"]    = norm.apply(lambda d: d["has_caps_brand"])
    work["__has_amp"]     = norm.apply(lambda d: d["ampersand"])

    # --- canonical only for non-fallbacks
    def canonical_for_group(g):
        tokens = g["__base_tokens"].dropna().tolist()
        base = next((t for t in tokens if t), [])
        prefer_caps = g["__has_caps"].mean() > 0.5
        amp = bool(g["__has_amp"].any())
        return build_canonical_from_tokens(base, prefer_caps, amp)

    canon = (
        work.loc[~work["__fallback"]]
            .groupby("__norm_key", dropna=False)
            .apply(canonical_for_group)
            .rename("canonical_affiliation")
            .reset_index()
    )

    out = work.merge(canon, on="__norm_key", how="left")

    # For fallback rows, just strip legal/US and keep raw form
    mask_fallback = out["__fallback"]
    out.loc[mask_fallback, "canonical_affiliation"] = (
        out.loc[mask_fallback, "organization"]
        .astype(str)  # convert NaN and others to string
        .apply(lambda x: " ".join(split_tokens_keep_case(to_ascii(x))))
    )

    merged = (
        out.groupby("canonical_affiliation", dropna=False)
           .agg(
               n_rows=("organization", "size"),
               sample_affiliations=("organization", lambda s: sorted(set(s))[:5]),
           )
           .reset_index()
           .sort_values(["n_rows","canonical_affiliation"], ascending=[False, True])
    )

    return out, merged

In [ ]:
df_canonical, log_canonical = standardize_affiliations(category_all)
df_canonical

In [ ]:
normalize_row_name('Ag Biotech Inc')

In [ ]:
log_canonical.head(20)

In [ ]:
df_canonical.to_parquet('04_canonical_firms_US.parquet')

In [ ]:
log_canonical.to_csv('04_canonical_log_US.csv')